# Train on all series + slow-lr param search

Uses the shipped 33-feat spectral StreamState (submit#1-features_improved). Two
levers together:
- **more training data**: 8000-series train / 2000-series holdout (vs the 6000 used
  before — the full 10k split into train+holdout),
- **slow learning rate**: the batch campaign noted lr .005 x ~3600 trees ≈ +0.006
  fast-CV. Compare the shipped fast params (lr .05, 500 trees) against slower
  configs, single-seed first, then 3-seed the winner.

Metric: full-res holdout TS-AUC. Watch train time — slow-lr configs are the cost.

In [1]:
import math, time
from bisect import bisect_left, bisect_right
from collections import deque
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import roc_auc_score
PATH = "/Users/minqi/Documents/ADIA_Lab_Structural_Break_Challenge/"

## StreamState — shipped 33-feat spectral set (verbatim from submission)

In [2]:

def _autocorr(x, lag):
    if len(x) <= lag: return 0.0
    a, b = x[:-lag], x[lag:]
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a*a).sum()*(b*b).sum())
    return float((a*b).sum()/d) if d > 0 else 0.0

def _fit_ar(x, p=5):
    if len(x) <= p+5: return None, 1.0
    rows = np.lib.stride_tricks.sliding_window_view(x, p+1)
    Y = rows[:, -1]; Xd = np.column_stack([np.ones(len(rows)), rows[:, :-1][:, ::-1]])
    beta, *_ = np.linalg.lstsq(Xd, Y, rcond=None); resid = Y - Xd @ beta
    return beta, float(resid.var()) if len(resid) else 1.0

def _norm_psd(seg, win):
    x = (seg - seg.mean()) * win; sp = np.fft.rfft(x)
    p = (sp.real**2 + sp.imag**2)[1:]; s = p.sum()
    if s <= 0 or not np.isfinite(s): return None
    return p / s

def _spec_ec(p, freqs, logk):
    return -float((p*np.log(p+1e-12)).sum())*logk, float((freqs*p).sum())


class StreamState:
    VAR_MIN = 20; SPEC_W = 128
    _BASE = ("n_online","mean_shift_z","last_z","online_logvar","online_std",
        "online_skew","online_kurt","cumsum_range","cusum_max","tail2_frac",
        "tail3_frac","jump_freq","signrun_freq","ks_stat","rank_dev","ac1_online",
        "ac1_shift","ar_resid_logratio","ref_log_n","ref_skew","ref_kurt","ref_ac1")
    _SPEC = ("spec_entropy","spec_entropy_shift","spec_centroid_shift")

    def __init__(self, bins=16, roll_windows=(50,200), ewma_alphas=(0.05,0.2), ar_p=5):
        self.bins=bins; self.rw=tuple(roll_windows); self.alphas=tuple(ewma_alphas); self.p=ar_p
        fn=list(self._BASE)
        for a in self.alphas: fn += ["ewma%.2f_z"%a,"ewma%.2f_logvar"%a]
        for w in self.rw: fn += ["roll%d_meanshift"%w,"roll%d_logvar"%w]
        self._ew_off=len(self._BASE); self._roll_off=self._ew_off+2*len(self.alphas)
        self._spec_off=len(fn); fn += list(self._SPEC)
        self.feature_names=fn; self.ncol=len(fn)
        self._out=np.zeros(self.ncol); self._bin_targets=np.linspace(1.0/self.bins,1.0,self.bins)
        W=self.SPEC_W; self._spec_win=np.hanning(W)
        self._spec_freqs=np.arange(1,W//2+1,dtype=np.float64)/(W//2); self._spec_logk=1.0/math.log(len(self._spec_freqs))

    def reset(self, reference):
        r=np.asarray(reference,dtype=np.float64)
        self.mu_h=float(r.mean()) if len(r) else 0.0
        self.sd_h=max(float(r.std(ddof=1)) if len(r)>1 else 1.0,1e-8); self.var_h=self.sd_h**2
        self.ref_n=len(r); self._inv_refn=1.0/max(self.ref_n,1)
        z=(r-self.mu_h)/self.sd_h if len(r) else r
        self.ref_skew=float((z**3).mean()) if len(r) else 0.0
        self.ref_kurt=float((z**4).mean()-3.0) if len(r) else 0.0
        self.ref_ac1=_autocorr(r,1); self.ref_log_n=math.log(self.ref_n+1.0)
        self.sorted_ref=np.sort(r).tolist() if len(r) else []
        edges=(np.quantile(r,np.linspace(0,1,self.bins+1)[1:-1]) if len(r) else np.zeros(self.bins-1))
        self.edges=edges.tolist(); self.ar,self.ar_var_h=_fit_ar(r,self.p)
        self.ref_spec_ent=0.0; self.ref_spec_cen=0.0
        if len(r)>=self.SPEC_W:
            W=self.SPEC_W; acc=np.zeros(W//2); cnt=0
            for s in range(0,len(r)-W+1,W//2):
                p=_norm_psd(r[s:s+W],self._spec_win)
                if p is not None: acc+=p; cnt+=1
            if cnt:
                pr=acc/acc.sum(); self.ref_spec_ent,self.ref_spec_cen=_spec_ec(pr,self._spec_freqs,self._spec_logk)
        self.n=0; self.mean=self.M2=self.M3=self.M4=0.0; self.sx=self.sxx=self.sxy=0.0
        self.cmax=self.cmin=self.cumsum=0.0; self.cusum_p=self.cusum_n=self.cusum_max=0.0
        self.tail2=self.tail3=0; self.binc=np.zeros(self.bins); self.rank_sum=0.0
        self.jumps=0; self.signruns=0; self.prev=None; self.ar_resid_ss=0.0; self.ar_n=0
        self.arbuf=deque(maxlen=self.p); self.roll={w:(deque(maxlen=w),[0.0,0.0]) for w in self.rw}
        self.ew={a:self.mu_h for a in self.alphas}; self.ewv={a:0.0 for a in self.alphas}
        self.specbuf=deque(maxlen=self.SPEC_W); self._last_spec_ent=0.0; self._last_spec_cen=0.0
        return self

    def update(self, x):
        x=float(x); self.n+=1; n=self.n; sd_h=self.sd_h; mu_h=self.mu_h; var_h=self.var_h
        d=x-self.mean; dn=d/n; dn2=dn*dn; term=d*dn*(n-1); self.mean+=dn
        self.M4+=term*dn2*(n*n-3*n+3)+6*dn2*self.M2-4*dn*self.M3
        self.M3+=term*dn*(n-2)-3*dn*self.M2; self.M2+=term
        var=self.M2/(n-1) if n>1 else 0.0; std=math.sqrt(var) if var>0 else 0.0; zx=(x-mu_h)/sd_h
        self.cumsum+=zx
        if self.cumsum>self.cmax: self.cmax=self.cumsum
        if self.cumsum<self.cmin: self.cmin=self.cumsum
        cp=self.cusum_p+zx-0.5; cn=self.cusum_n-zx-0.5
        self.cusum_p=cp if cp>0 else 0.0; self.cusum_n=cn if cn>0 else 0.0
        if self.cusum_p>self.cusum_max: self.cusum_max=self.cusum_p
        if self.cusum_n>self.cusum_max: self.cusum_max=self.cusum_n
        azx=zx if zx>=0 else -zx
        if azx>2.0: self.tail2+=1
        if azx>3.0: self.tail3+=1
        if self.prev is not None:
            if abs(x-self.prev)>2.0*sd_h: self.jumps+=1
            if (x-mu_h)*(self.prev-mu_h)<0: self.signruns+=1
            self.sxy+=x*self.prev
        self.sx+=x; self.sxx+=x*x
        lo=bisect_left(self.sorted_ref,x); hi=bisect_right(self.sorted_ref,x)
        self.rank_sum+=0.5*(lo+hi)*self._inv_refn; self.binc[bisect_right(self.edges,x)]+=1
        if self.ar is not None and len(self.arbuf)==self.p:
            lag=np.array(self.arbuf,dtype=np.float64)[::-1]
            pred=self.ar[0]+self.ar[1:]@lag; r_=x-pred; self.ar_resid_ss+=r_*r_; self.ar_n+=1
        self.arbuf.append(x)
        for a in self.alphas:
            m0=self.ew[a]; self.ew[a]=(1-a)*m0+a*x; self.ewv[a]=(1-a)*(self.ewv[a]+a*(x-m0)**2)
        for w,(buf,acc) in self.roll.items():
            if len(buf)==buf.maxlen: old=buf[0]; acc[0]-=old; acc[1]-=old*old
            buf.append(x); acc[0]+=x; acc[1]+=x*x
        self.specbuf.append(x)
        if len(self.specbuf)==self.SPEC_W:
            p=_norm_psd(np.array(self.specbuf,dtype=np.float64),self._spec_win)
            if p is not None: self._last_spec_ent,self._last_spec_cen=_spec_ec(p,self._spec_freqs,self._spec_logk)
        self.prev=x
        out=self._out; se=sd_h/math.sqrt(n); emp=np.cumsum(self.binc)/n
        ks=float(np.max(np.abs(emp-self._bin_targets)))
        if n>2 and var>0:
            mx=self.sx/n; cov=self.sxy/(n-1)-mx*mx*n/(n-1); ac1=cov/var
        else: ac1=0.0
        vg=n>=self.VAR_MIN; sqrt_n=math.sqrt(n)
        out[0]=math.log1p(n); out[1]=(self.mean-mu_h)/max(se,1e-8); out[2]=zx
        out[3]=math.log((var+1e-8)/var_h) if vg else 0.0; out[4]=std/sd_h
        out[5]=(self.M3/n)/(var**1.5+1e-8) if vg else 0.0
        out[6]=((self.M4/n)/(var**2+1e-8)-3.0) if vg else 0.0
        out[7]=(self.cmax-self.cmin)/sqrt_n; out[8]=self.cusum_max/sqrt_n
        out[9]=self.tail2/n; out[10]=self.tail3/n; out[11]=self.jumps/n; out[12]=self.signruns/n
        out[13]=ks; out[14]=self.rank_sum/n-0.5; out[15]=ac1; out[16]=ac1-self.ref_ac1
        out[17]=(math.log((self.ar_resid_ss/self.ar_n+1e-8)/(self.ar_var_h+1e-8)) if self.ar_n>=self.VAR_MIN else 0.0)
        out[18]=self.ref_log_n; out[19]=self.ref_skew; out[20]=self.ref_kurt; out[21]=self.ref_ac1
        off=self._ew_off
        for ai,a in enumerate(self.alphas):
            ew_se=sd_h*math.sqrt(a/(2-a))
            out[off+2*ai]=(self.ew[a]-mu_h)/max(ew_se,1e-8)
            out[off+2*ai+1]=math.log((self.ewv[a]+1e-8)/var_h) if vg else 0.0
        off=self._roll_off
        for wi,w in enumerate(self.rw):
            buf,acc=self.roll[w]; L=len(buf); m=acc[0]/L; v=acc[1]/L-m*m
            if v<0: v=0.0
            out[off+2*wi]=(m-mu_h)/sd_h; out[off+2*wi+1]=math.log((v+1e-8)/var_h)
        off=self._spec_off
        out[off]=self._last_spec_ent; out[off+1]=self._last_spec_ent-self.ref_spec_ent
        out[off+2]=self._last_spec_cen-self.ref_spec_cen
        return out

def _log_steps(L,m):
    if L<=m: return np.arange(L)
    return np.unique(np.round(np.expm1(np.linspace(0,np.log1p(L-1),m))).astype(int))

print("StreamState feats:", StreamState().ncol)

StreamState feats: 33


## Load + split (8000 train / 2000 holdout — the full 10k)

In [3]:
X=pd.read_parquet(PATH+"X_train.parquet"); Yi=pd.read_parquet(PATH+"y_train_index.parquet")
tau_map=Yi["tau_index"].to_dict()
series=[]
for sid,sub in X.groupby(level="id",sort=False):
    v=sub["value"].to_numpy(); per=sub["period"].to_numpy(); t=int(tau_map[sid])
    series.append((sid,v[per==1],v[per==2],None if t==-1 else t))
train_series=series[:8000]; hold_series=series[8000:10000]
print("total",len(series),"train",len(train_series),"hold",len(hold_series))

total 10000 train 8000 hold 2000


## Extract (8000 subsample train, 2000 full-res holdout)

In [4]:
SAMPLES_PER=40
def extract(series_list, full):
    st=StreamState(); rows=[]; lab=[]; step=[]; sid_=[]
    for sid,xh,xo,tau in series_list:
        st.reset(xh); L=len(xo)
        want=None if full else set(_log_steps(L,SAMPLES_PER).tolist())
        for k in range(L):
            out=st.update(xo[k])
            if full or k in want:
                rows.append(out.copy()); lab.append(1 if (tau is not None and k>=tau) else 0)
                step.append(k); sid_.append(sid)
    return np.asarray(rows,np.float32),np.asarray(lab,np.int8),np.asarray(step,np.int32),np.asarray(sid_)
t0=time.time(); Xtr,ytr,_,sids=extract(train_series,False); print("train",Xtr.shape,"%.0fs"%(time.time()-t0))
t0=time.time(); Xho,yho,steps,_=extract(hold_series,True); print("hold",Xho.shape,"%.0fs"%(time.time()-t0))
cmap=dict(zip(*np.unique(sids,return_counts=True)))
W_TR=np.array([1.0/cmap[s] for s in sids]); W_TR/=W_TR.mean()

train (263271, 33) 126s


hold (1001880, 33) 35s


## Param configs + evaluator

In [5]:
BASE=dict(objective="binary",num_leaves=63,min_child_samples=300,subsample=0.8,
    subsample_freq=1,colsample_bytree=0.8,reg_lambda=1.0,n_jobs=-1,verbosity=-1)
CONFIGS={
  "fast_lr05_500  (shipped)": dict(learning_rate=0.05, n_estimators=500),
  "med_lr02_1500":            dict(learning_rate=0.02, n_estimators=1500),
  "slow_lr01_3000":           dict(learning_rate=0.01, n_estimators=3000),
  "slow_lr005_5000":          dict(learning_rate=0.005, n_estimators=5000),
}
def ts_auc(y,sc,st):
    num=den=0.0
    for t in np.unique(st):
        m=st==t; yv=y[m]; npos=int(yv.sum()); nneg=len(yv)-npos
        if npos==0 or nneg==0: continue
        num+=npos*nneg*roc_auc_score(yv,sc[m]); den+=npos*nneg
    return num/den if den else 0.5
def fit_eval(params, seed=0):
    t0=time.time()
    m=lgb.LGBMClassifier(random_state=seed,**BASE,**params).fit(Xtr,ytr,sample_weight=W_TR)
    a=ts_auc(yho,m.predict_proba(Xho)[:,1],steps)
    return a, time.time()-t0

## Single-seed sweep

In [6]:
res=[]
for name,p in CONFIGS.items():
    a,dt=fit_eval(p); res.append((name,a,dt)); print("%-26s TS-AUC %.4f  (%.0fs)"%(name,a,dt))
res_df=pd.DataFrame(res,columns=["config","ts_auc","fit_s"]).sort_values("ts_auc",ascending=False)
res_df.reset_index(drop=True)

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


fast_lr05_500  (shipped)   TS-AUC 0.5781  (16s)


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


med_lr02_1500              TS-AUC 0.5826  (54s)


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


slow_lr01_3000             TS-AUC 0.5813  (137s)


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


slow_lr005_5000            TS-AUC 0.5835  (249s)


,config,ts_auc,fit_s
0,slow_lr005_5000,0.583507,249.204966
1,med_lr02_1500,0.582627,54.315752
2,slow_lr01_3000,0.581292,137.404322
3,fast_lr05_500 (shipped),0.578075,16.317133


## 3-seed bag of the best config

In [7]:
best=res_df.iloc[0]["config"]; bp=CONFIGS[best]
print("bagging:",best,bp)
sc=np.zeros(len(Xho))
for s in range(3):
    m=lgb.LGBMClassifier(random_state=s,**BASE,**bp).fit(Xtr,ytr,sample_weight=W_TR)
    sc+=m.predict_proba(Xho)[:,1]
sc/=3
print("3-seed TS-AUC = %.4f"%ts_auc(yho,sc,steps))
print("(shipped fast 3-seed on 6000/2000 split was 0.5934; this split differs)")

bagging: slow_lr005_5000 {'learning_rate': 0.005, 'n_estimators': 5000}


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


3-seed TS-AUC = 0.5829
(shipped fast 3-seed on 6000/2000 split was 0.5934; this split differs)
